### Preliminaries

In [ ]:
import time
from operator import itemgetter
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

### Prepare Dataset

In [ ]:
def load_titanic():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    X = df.drop('survived', axis=1)
    y = df['survived']
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=201)

X_train, X_test, y_train, y_test = load_titanic()
scaler = StandardScaler().fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)
X_train.head(n=5)

In [ ]:
NUM_MODELS = 20

In [ ]:
def train_and_score_model(train_set, test_set, train_labels, test_labels, n_estimators):
    start_time = time.time()
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=201)
    model.fit(train_set, train_labels)
    score = accuracy_score(test_labels, model.predict(test_set))
    delta = time.time() - start_time
    print(f"n_estimators={n_estimators}, accuracy={score:.4f}, took: {delta:.2f}s")
    return n_estimators, score

In [ ]:
def run_sequential(n_models):
    return [
        train_and_score_model(X_train, X_test, y_train, y_test, 8 + 4*j)
        for j in range(n_models)
    ]

This function trains `n_models` sequentially for increasing `n_estimators` (8, 12, 16, ...). Returns `[(n_estimators, accuracy), ...]`.

In [ ]:
%%time
accuracy_scores = run_sequential(n_models=NUM_MODELS)

### Analyze Results

In [ ]:
best = max(accuracy_scores, key=itemgetter(1))
print(f"Best model: accuracy={best[1]:.4f}, n_estimators={best[0]}")

Note how long training NUM_MODELS sequentially took. Continue to the next section to learn how to improve runtime by distributing this task.

### Parallel Implementation

In contrast to the previous approach, you will now utilize all available resources to train these models in parallel. Ray will automatically detect the number of cores on your computer and distribute each task.

In [ ]:
# !pip install ray

In [ ]:
import ray

if ray.is_initialized():
    ray.shutdown()

ray.init()

Workers use `ray.put()` to place objects and `ray.get()` to retrieve them from each node's object store. These object stores form the shared distributed memory that makes objects available across a Ray cluster.

In a distributed system, object references are pointers to objects in memory. They allow different workers to access the same data without copying it.

In [ ]:
X_train_ref = ray.put(X_train)
X_test_ref  = ray.put(X_test)
y_train_ref = ray.put(y_train)
y_test_ref  = ray.put(y_test)

By placing training and testing data into Ray's object store, these objects are now available to all remote tasks in the cluster.

**Coding Exercise**: Print `X_train_ref` and retrieve `X_train` using `ray.get()`.

In [ ]:
### SAMPLE IMPLEMENTATION ###
print(X_train_ref)
ray.get(X_train_ref)

### Implement Function to Train and Score Model

In [ ]:
@ray.remote
def train_and_score_model(train_set_ref, test_set_ref, train_labels_ref, test_labels_ref, n_estimators):
    start_time = time.time()
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=201)
    model.fit(train_set_ref, train_labels_ref)
    score = accuracy_score(test_labels_ref, model.predict(test_set_ref))
    delta = time.time() - start_time
    print(f"n_estimators={n_estimators}, accuracy={score:.4f}, took: {delta:.2f}s")
    return n_estimators, score

Notice that `train_and_score_model` is the same function as in the sequential example, except with the `@ray.remote` decorator to specify distributed execution.

**Implement function that runs parallel model training**

In [ ]:
def run_parallel(n_models):
    results_ref = [
        train_and_score_model.remote(
            train_set_ref=X_train_ref, test_set_ref=X_test_ref,
            train_labels_ref=y_train_ref, test_labels_ref=y_test_ref,
            n_estimators=8 + 4*j,
        )
        for j in range(n_models)
    ]
    return ray.get(results_ref)

### Run Parallel Model Training

In [ ]:
%%time
accuracy_scores = run_parallel(n_models=NUM_MODELS)

### Notice Performance Gain:
- Parallel: ~10s
- Sequential: ~60s

### Analyze Results

In [ ]:
best = max(accuracy_scores, key=itemgetter(1))
print(f"Best model: accuracy={best[1]:.4f}, n_estimators={best[0]}")

In [ ]:
ray.shutdown()